# GFP 결합체 설계와 형광 변화 예측 · v0.1.3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/GFP_Binder_T4.ipynb?v=0.1.3)

T4 GPU 런타임에서 전체 실행을 선택합니다. 뼈대 2개에 각각 서열 2개를 설계하고 GFP 복합체를 하나씩 예측합니다. 기본 대조군 두 개로는 복합체 구조를 평가하고 형광 배율은 보류합니다.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="gfp_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")

## 1–4. 설계와 예측

자체 실험 데이터가 있으면 Colab 파일 패널에 CSV를 올리고 아래 경로를 입력합니다. CSV에는 `id,ratio,group,assay,source`와 `sequence` 또는 `pdb,entity`가 필요합니다. `ratio`는 동일 실험에서 측정한 기준 대비 형광 배율이고, `group`은 유사 결합체의 그룹입니다. 경로를 비워 두면 공개 대조군을 씁니다.

In [ ]:
ASSAY_CSV = ""  # @param {type:"string"}
command = [sys.executable, str(checkout / "gfp_pipeline/run.py"), "--revision", revision]
if ASSAY_CSV.strip():
    command.extend(["--assays", str(Path(ASSAY_CSV).expanduser().resolve(strict=True))])
with subprocess.Popen(command, stdout=subprocess.PIPE, text=True) as process:
    for line in process.stdout:
        print(line.rstrip())
        if line.rstrip().endswith(".zip"):
            result_zip = Path(line.strip())
    if process.wait():
        raise RuntimeError(f"Pipeline failed with exit code {process.returncode}.")
result_dir = result_zip.with_suffix("")

## 후보별 결과

`iptm`은 예측된 접촉면의 구조 신뢰도입니다. 형광 배율 1.0은 기준과 같은 형광입니다.

In [ ]:
import json
import pandas as pd

metrics = json.loads((result_dir / "metrics/fluorescence_model.json").read_text())
status_text = {
    "ready": "형광 모델 검증 통과",
    "structural_benchmark_failed": "대조군 구조 검증 미통과",
    "insufficient_control_groups": "독립 실험 그룹 부족",
    "baseline_not_improved": "형광 변화 없음 기준 대비 개선 없음",
    "candidate_structure_failed": "후보 구조 검증 미통과",
    "outside_training_range": "접촉 특징이 학습 범위 밖",
    "outside_observed_effect_range": "예측값이 실험에서 관측한 범위 밖",
    "predicted": "예측값 산출",
}
print(status_text[metrics['validation']])
if "held_out_mae_log2" in metrics:
    print(f"그룹별 검증 MAE (log2): {metrics['held_out_mae_log2']:.3f}")
    print(f"변화 없음 기준 MAE (log2): {metrics['no_effect_mae_log2']:.3f}")

results = pd.read_csv(result_dir / "metrics/fluorescence_predictions.csv")
results["prediction_status"] = results["prediction_status"].map(status_text)
ratio = results["predicted_fluorescence_ratio"].round(3)
results["predicted_fluorescence_ratio"] = ratio.astype(object).where(ratio.notna(), "보류")
display(results[["id", "iptm", "prediction_status", "predicted_fluorescence_ratio"]])

In [ ]:
from google.colab import files

files.download(str(result_zip))